# Direct Multi-Step — Decision Tree baseline (10 model)


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [3]:
def get_file_path(filename):
    current_dir = Path.cwd()
    for search_root in [current_dir] + list(current_dir.parents):
        for file in search_root.rglob(filename):
            if file.is_file():
                return file
    raise FileNotFoundError(f"Không tìm thấy file {filename}!")


train_df = pd.read_csv(get_file_path("train_final.csv"))
val_df = pd.read_csv(get_file_path("val_set.csv"))

for df in (train_df, val_df):
    df["Date"] = pd.to_datetime(df["Date"])

print("train :", train_df.shape, "|", train_df["Date"].min().date(), "->", train_df["Date"].max().date())
print("val   :", val_df.shape, "|", val_df["Date"].min().date(), "->", val_df["Date"].max().date())

train : (2700, 24) | 2011-02-04 -> 2012-03-23
val   : (1395, 24) | 2012-03-30 -> 2012-10-26


In [9]:
HORIZON = 10


def add_targets(df, horizon=HORIZON):
    d = df.sort_values(["Store", "Date"]).reset_index(drop=True).copy()
    for h in range(1, horizon + 1):
        d[f"target_t+{h}"] = d.groupby("Store")["Weekly_Sales"].shift(-h)
    return d


train_set = add_targets(train_df)
val_set = add_targets(val_df)

target_cols = [f"target_t+{h}" for h in range(1, HORIZON + 1)]
# visual trên store 1, bảng target
display(train_set.loc[train_set["Store"] == 1, ["Store", "Date", "Weekly_Sales"] + target_cols[:10]].head(8))
print(train_set.columns)

,Store,Date,Weekly_Sales,target_t+1,target_t+2,target_t+3,target_t+4,target_t+5,target_t+6,target_t+7,target_t+8,target_t+9,target_t+10
0,1,2011-02-04,1606629.58,1649614.93,1686842.78,1456800.28,1636263.41,1553191.63,1576818.06,1541102.38,1495064.75,1614259.35,1559889.00
1,1,2011-02-11,1649614.93,1686842.78,1456800.28,1636263.41,1553191.63,1576818.06,1541102.38,1495064.75,1614259.35,1559889.00,1564819.81
2,1,2011-02-18,1686842.78,1456800.28,1636263.41,1553191.63,1576818.06,1541102.38,1495064.75,1614259.35,1559889.00,1564819.81,1455090.69
3,1,2011-02-25,1456800.28,1636263.41,1553191.63,1576818.06,1541102.38,1495064.75,1614259.35,1559889.00,1564819.81,1455090.69,1629391.28
4,1,2011-03-04,1636263.41,1553191.63,1576818.06,1541102.38,1495064.75,1614259.35,1559889.00,1564819.81,1455090.69,1629391.28,1604775.58
5,1,2011-03-11,1553191.63,1576818.06,1541102.38,1495064.75,1614259.35,1559889.00,1564819.81,1455090.69,1629391.28,1604775.58,1428218.27
6,1,2011-03-18,1576818.06,1541102.38,1495064.75,1614259.35,1559889.00,1564819.81,1455090.69,1629391.28,1604775.58,1428218.27,1466046.67
7,1,2011-03-25,1541102.38,1495064.75,1614259.35,1559889.00,1564819.81,1455090.69,1629391.28,1604775.58,1428218.27,1466046.67,1635078.41


Index(['Store', 'Date', 'IsHoliday', 'Weekly_Sales', 'Type', 'Size',
       'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3',
       'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Lag_1', 'Lag_4',
       'Lag_12', 'Lag_52', 'Rolling_Mean_4w', 'Year', 'Month', 'WeekOfYear',
       'Type_encoded', 'target_t+1', 'target_t+2', 'target_t+3', 'target_t+4',
       'target_t+5', 'target_t+6', 'target_t+7', 'target_t+8', 'target_t+9',
       'target_t+10'],
      dtype='str')


In [11]:
drop_cols = ["Date", "Weekly_Sales", "Type",] + target_cols
feature_cols = [c for c in train_set.columns if c not in drop_cols]

print(f"{len(feature_cols)} cột feature:")
print(feature_cols)

obj_cols = [c for c in feature_cols if train_set[c].dtype == object]
assert not obj_cols, f"Còn cột chuỗi trong feature: {obj_cols}"

21 cột feature:
['Store', 'IsHoliday', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Lag_1', 'Lag_4', 'Lag_12', 'Lag_52', 'Rolling_Mean_4w', 'Year', 'Month', 'WeekOfYear', 'Type_encoded']


In [12]:
# giữ đúng tham số tương ứng bản RF để so được: khác biệt chỉ nằm ở 1 cây vs 200 cây
dt_params = dict(max_depth=12, min_samples_split=5, min_samples_leaf=2, random_state=42)

dt_models, metrics, pred_frames = {}, [], []

for h in range(1, HORIZON + 1):
    target = f"target_t+{h}"
    train_c = train_set.dropna(subset=[target])
    val_c   = val_set.dropna(subset=[target])
    X_train, y_train = train_c[feature_cols], train_c[target]
    X_val,   y_val   = val_c[feature_cols],   val_c[target]

    dt = DecisionTreeRegressor(**dt_params).fit(X_train, y_train)
    preds = dt.predict(X_val)
    dt_models[h] = dt

    pred_frames.append(pd.DataFrame({
        "Store": val_c["Store"],
        "Date": val_c["Date"],
        "target_Date": val_c["Date"] + pd.Timedelta(days=h * 7),
        "horizon": h,
        "y_true": y_val,
        "y_pred": preds
    }))

    mae  = mean_absolute_error(y_val, preds)
    rmse = root_mean_squared_error(y_val, preds)
    wape = np.abs(y_val - preds).sum() / np.abs(y_val).sum()
    metrics.append({"horizon": f"t+{h:02d}", "n_train": len(train_c), "n_val": len(val_c),
                    "RMSE": rmse, "WAPE": wape, "MAE": mae})
    print(f"t+{h:02d} | RMSE {rmse:10,.0f} | WAPE {wape:6.2%} | MAE {mae:10,.0f}")

predictions = pd.concat(pred_frames, ignore_index=True)
metrics_df  = pd.DataFrame(metrics).set_index("horizon")

t+01 | RMSE     98,182 | WAPE  6.49% | MAE     67,644
t+02 | RMSE    247,033 | WAPE 12.19% | MAE    126,481
t+03 | RMSE    105,322 | WAPE  6.66% | MAE     69,099
t+04 | RMSE    125,242 | WAPE  7.16% | MAE     74,398
t+05 | RMSE    114,139 | WAPE  7.10% | MAE     73,996
t+06 | RMSE    116,556 | WAPE  6.89% | MAE     71,778
t+07 | RMSE    115,946 | WAPE  7.22% | MAE     75,139
t+08 | RMSE    119,684 | WAPE  7.69% | MAE     80,061
t+09 | RMSE    134,267 | WAPE  8.85% | MAE     92,048
t+10 | RMSE    147,999 | WAPE  9.03% | MAE     93,765


In [13]:
view = metrics_df.copy()
view["WAPE"] = (view["WAPE"] * 100).round(2).astype(str) + "%"
display(view.round({"RMSE": 0, "MAE": 0}))

,n_train,n_val,RMSE,WAPE,MAE
horizon,,,,,
t+01,2655,1350,98182.0,6.49%,67644.0
t+02,2610,1305,247033.0,12.19%,126481.0
t+03,2565,1260,105322.0,6.66%,69099.0
t+04,2520,1215,125242.0,7.16%,74398.0
t+05,2475,1170,114139.0,7.1%,73996.0
t+06,2430,1125,116556.0,6.89%,71778.0
t+07,2385,1080,115946.0,7.22%,75139.0
t+08,2340,1035,119684.0,7.69%,80061.0
t+09,2295,990,134267.0,8.85%,92048.0


### So sánh giữa các horizon
Vì mỗi h thì cái tập val nó khác nhau, nên đánh giá chung trên 21 tuần dữ liệu. Sẽ test được ful hết cùng sample với nhau

In [28]:
all_dates = sorted(val_set["Date"].unique())
common_dates = all_dates[:-HORIZON]
common_df = predictions[predictions["Date"].isin(common_dates)].copy()

records = []
for h in range(1, HORIZON + 1):
    sub = common_df[common_df["horizon"] == h]
    y_true, y_pred = sub["y_true"], sub["y_pred"]
    records.append({
        "Horizon": f"t+{h:02d}",
        "Số dòng": len(sub),
        "RMSE": f"{root_mean_squared_error(y_true, y_pred):,.0f}",
        "WAPE": f"{np.abs(y_true - y_pred).sum() / np.abs(y_true).sum() * 100:.2f}%",
        "MAE": f"{mean_absolute_error(y_true, y_pred):,.0f}"
    })

print(f"Đánh giá trên {len(common_dates)} tuần gốc: "
      f"{pd.to_datetime(common_dates[0]).date()} -> {pd.to_datetime(common_dates[-1]).date()}")
print()
display(pd.DataFrame(records).set_index("Horizon"))

Đánh giá trên 21 tuần gốc: 2012-03-30 -> 2012-08-17



,Số dòng,RMSE,WAPE,MAE
Horizon,,,,
t+01,945,"101,159",6.61%,"69,670"
t+02,945,"221,118",10.94%,"114,477"
t+03,945,"108,135",6.70%,"70,223"
t+04,945,"123,623",7.18%,"75,248"
t+05,945,"117,784",7.26%,"76,139"
t+06,945,"113,925",6.83%,"71,350"
t+07,945,"119,538",7.40%,"77,336"
t+08,945,"121,817",7.77%,"81,129"
t+09,945,"135,688",8.95%,"93,216"
